In [9]:
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report 


In [10]:
driver_safety_model =joblib.load("../MODELS/driver_safety_model.pkl")
breakdown_model=joblib.load("../MODELS/breakdown_model.pkl")
print(" model loaded successfully")

 model loaded successfully


In [11]:
df=pd.read_csv("../DATA/processed/truck_processed.csv")
print("Dataset shape:",df.shape)
print(df.head())

Dataset shape: (5000, 19)
   Driver_Age  Driving_Hours  Speed  Harsh_Braking  Harsh_Acceleration  \
0          50            5.4     47              2                   1   
1          36            9.3     69              1                   0   
2          29            6.0     81              1                   2   
3          42            5.5     62              1                   4   
4          40           11.3     55              0                   0   

   Drowsiness_Score  Engine_Temperature  Engine_Vibration  Battery_Voltage  \
0              65.7                83.1              1.76            13.08   
1             100.0                97.2              3.68            11.88   
2              70.0                93.8              1.98            12.30   
3              62.2                84.9              4.25            12.90   
4             100.0                94.0              4.85            12.56   

   Oil_Pressure  Tyre_Pressure  Vehicle_Age  Mileage  Previo

In [12]:
from sklearn.model_selection import train_test_split

## features and driver safety target

X=df.drop(columns=["Driver_Risk","Breakdown_Risk"])
y_driver=df["Driver_Risk"]

## train test split for verification 
X_train,X_test,y_train,y_test=train_test_split(X,y_driver,test_size=0.20,random_state=42,stratify=y_driver)


## prediction using saved model

driver_pred= driver_safety_model.predict(X_test)

## Evaluation

driver_accuracy=accuracy_score(y_test,driver_pred)
driver_precision=precision_score(y_test,driver_pred)
driver_recall=recall_score(y_test,driver_pred)
driver_f1=f1_score(y_test,driver_pred)

print("DRIVER SAFETY MODEL VERIFICATION ")
print(f"Accuracy : {driver_accuracy:.4f}")
print(f"Precision: {driver_precision:.4f}")
print(f"Recall   : {driver_recall:.4f}")
print(f"F1 Score : {driver_f1:.4f}")


DRIVER SAFETY MODEL VERIFICATION 
Accuracy : 0.9880
Precision: 1.0000
Recall   : 0.9703
F1 Score : 0.9849


In [13]:
## features and breakdown risk target
x_breakdown=df.drop(columns=["Driver_Risk","Breakdown_Risk"])
y_breakdown=df["Breakdown_Risk"]


## train test split

X_train_b,X_test_b,y_train_b,y_test_b=train_test_split(x_breakdown,y_breakdown,test_size=0.20,random_state=42,stratify=y_breakdown)

## get high risk probabilities

breakdown_probability= breakdown_model.predict_proba(X_test_b)[:,1]



# Apply final threshold
best_threshold = 0.40

final_breakdown_pred = ( breakdown_probability >= best_threshold).astype(int)

# Evaluation
breakdown_accuracy = accuracy_score(y_test_b,final_breakdown_pred)

breakdown_precision = precision_score(y_test_b,final_breakdown_pred)

breakdown_recall = recall_score(y_test_b,final_breakdown_pred)

breakdown_f1 = f1_score(y_test_b,final_breakdown_pred)

missed_high_risk = sum((y_test_b == 1) &(final_breakdown_pred == 0))

print(" BREAKDOWN MODEL FINAL VERIFICATION ")
print(f"Threshold : {best_threshold}")
print(f"Accuracy  : {breakdown_accuracy:.4f}")
print(f"Precision : {breakdown_precision:.4f}")
print(f"Recall    : {breakdown_recall:.4f}")
print(f"F1 Score  : {breakdown_f1:.4f}")
print(f"Missed High-Risk Cases : {missed_high_risk}")

 BREAKDOWN MODEL FINAL VERIFICATION 
Threshold : 0.4
Accuracy  : 0.9770
Precision : 0.8101
Recall    : 0.8889
F1 Score  : 0.8477
Missed High-Risk Cases : 8


In [14]:
final_model_config = {
    "breakdown_threshold": best_threshold,
    "driver_safety_model": "driver_safety_model.pkl",
    "breakdown_model": "breakdown_model.pkl"
}

joblib.dump(
    final_model_config,
    "../models/model_config.pkl"
)

print("Final model configuration saved successfully ✅")
print(final_model_config)

Final model configuration saved successfully ✅
{'breakdown_threshold': 0.4, 'driver_safety_model': 'driver_safety_model.pkl', 'breakdown_model': 'breakdown_model.pkl'}


In [15]:
# Select one sample truck from the dataset
sample_truck = X_test.iloc[[0]]

# Driver Safety prediction
driver_risk_prediction = driver_safety_model.predict(sample_truck)[0]
driver_risk_probability = driver_safety_model.predict_proba(
    sample_truck
)[0, 1]

# Breakdown prediction
breakdown_probability = breakdown_model.predict_proba(
    sample_truck
)[0, 1]

breakdown_risk_prediction = int(
    breakdown_probability >= best_threshold
)

# Convert prediction to labels
driver_risk_label = (
    "High Risk" if driver_risk_prediction == 1 else "Low Risk"
)

breakdown_risk_label = (
    "High Risk" if breakdown_risk_prediction == 1 else "Low Risk"
)

print("FINAL SAMPLE TRUCK PREDICTION ")

print("\nDriver Safety Prediction")
print("Risk Level      :", driver_risk_label)
print(
    "High-Risk Probability:",
    f"{driver_risk_probability:.4f}"
)

print("\nBreakdown Prediction")

print("Risk Level      :", breakdown_risk_label)
print(
    "High-Risk Probability:",
    f"{breakdown_probability:.4f}"
)
print("Applied Threshold:", best_threshold)

FINAL SAMPLE TRUCK PREDICTION 

Driver Safety Prediction
Risk Level      : Low Risk
High-Risk Probability: 0.0900

Breakdown Prediction
Risk Level      : Low Risk
High-Risk Probability: 0.0000
Applied Threshold: 0.4


In [16]:
print("========== FINAL MODEL SETUP SUMMARY ==========")

print("\n🚗 Driver Safety Model")
print("----------------------")
print(f"Accuracy  : {driver_accuracy:.4f}")
print(f"Precision : {driver_precision:.4f}")
print(f"Recall    : {driver_recall:.4f}")
print(f"F1 Score  : {driver_f1:.4f}")

print("\n🔧 Breakdown Risk Model")
print("-----------------------")
print(f"Accuracy  : {breakdown_accuracy:.4f}")
print(f"Precision : {breakdown_precision:.4f}")
print(f"Recall    : {breakdown_recall:.4f}")
print(f"F1 Score  : {breakdown_f1:.4f}")
print(f"Threshold : {best_threshold}")

print("\n📁 Final Model Files")
print("--------------------")
print("1. driver_safety_model.pkl")
print("2. breakdown_model.pkl")
print("3. driver_preprocessor.pkl")
print("4. vehicle_preprocessor.pkl")
print("5. model_config.pkl")

print("\n========== PROJECT STATUS ==========")
print("Model Training        : COMPLETE ✅")
print("Model Evaluation      : COMPLETE ✅")
print("Model Improvement     : COMPLETE ✅")
print("Final Verification    : COMPLETE ✅")
print("Deployment Ready      : YES 🚀")

========== FINAL MODEL SETUP SUMMARY ==========

🚗 Driver Safety Model
----------------------
Accuracy  : 0.9880
Precision : 1.0000
Recall    : 0.9703
F1 Score  : 0.9849

🔧 Breakdown Risk Model
-----------------------
Accuracy  : 0.9770
Precision : 0.8101
Recall    : 0.8889
F1 Score  : 0.8477
Threshold : 0.4

📁 Final Model Files
--------------------
1. driver_safety_model.pkl
2. breakdown_model.pkl
3. driver_preprocessor.pkl
4. vehicle_preprocessor.pkl
5. model_config.pkl

========== PROJECT STATUS ==========
Model Training        : COMPLETE ✅
Model Evaluation      : COMPLETE ✅
Model Improvement     : COMPLETE ✅
Final Verification    : COMPLETE ✅
Deployment Ready      : YES 🚀
